In [2]:
import torch
import json
import os
from transformers import AutoTokenizer, BertModel, Wav2Vec2Model
# from utils.audio_processing import AudioProcessor
import torchaudio
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

In [ ]:
from safetensors.torch import load_file
try:
    # **加载 safetensors**
    # 本地加载
    safetensors_path = "E:\Github项目\web\model.safetensors"
    # huggingface_hub 仓库下载
    # model_path = hf_hub_download(repo_id="liloge/Group7_model_test", filename="model.safetensors")
    state_dict = load_file(safetensors_path)
    
    # **检查是否正确加载**
    print(f"成功加载 `{safetensors_path}`，包含 {len(state_dict.keys())} 个参数")
    
    # **打印部分参数**
    for key in list(state_dict.keys()):
        print(f" - {key}: {state_dict[key].shape}")
    
    # **确保权重匹配**
    # model.load_state_dict(state_dict, strict=False)

except Exception as e:
    print(f"❌ 加载 `safetensors` 失败: {e}")
    exit(1)

In [1]:
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoConfig, Wav2Vec2ForPreTraining

class MultimodalClassifier(nn.Module):
    def __init__(self, wav2vec2_config_path):
        super().__init__()
        
        # **加载微调后的 BERT**
        self.bert = AutoModelForSequenceClassification.from_pretrained(
            "bert-base-uncased", num_labels=7
        )
        self.bert.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self.bert.config.hidden_size, self.bert.config.num_labels)
        )
        # try:
        #     self.bert.load_state_dict(torch.load(bert_ckpt_path, map_location=torch.device("cpu")), strict=True)
        # except Exception as e:
        #     print(f"❌ 加载 `{bert_ckpt_path}` 失败: {e}")
            
        # **先加载 Wav2Vec2**
        config = AutoConfig.from_pretrained(wav2vec2_config_path, num_labels=7)
        self.wav2vec2 = Wav2Vec2ForPreTraining.from_pretrained("facebook/wav2vec2-base", config=config)

        # **再修改 Wav2Vec2 的分类头**
        self.wav2vec2.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self.wav2vec2.config.hidden_size, self.wav2vec2.config.num_labels)
        )
        # # **加载 safetensors 权重**
        # from safetensors.torch import load_file
        # state_dict = load_file(wav2vec2_safetensors_path)
        # try:
        #     self.wav2vec2.load_state_dict(state_dict, strict=False)
        # except Exception as e:
        #     print(f"❌ 加载 `{wav2vec2_safetensors_path}` 失败: {e}")

        # **拼接特征的分类头**
        self.classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size + self.wav2vec2.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.7),
            nn.Linear(256, 7)  # 7分类任务
        )

    def forward(self, text_input, audio_input):
        # **文本特征**
        text_outputs = self.bert(**text_input, output_hidden_states=True)
        text_features = text_outputs.hidden_states[-1][:, 0, :]

        # **音频特征**
        audio_outputs = self.wav2vec2(audio_input, output_hidden_states=True)
        audio_features = audio_outputs.hidden_states[-1][:, 0, :]

        # **拼接特征**
        combined_features = torch.cat((text_features, audio_features), dim=-1)

        # **分类**
        logits = self.classifier(combined_features)
        return logits


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# **定义路径**
# bert_ckpt_path = "bert_meld_finetune_model.pth"
wav2vec2_config_path = "config.json"
# wav2vec2_safetensors_path = "wav2vec2.safetensors"

# **加载模型**
model = MultimodalClassifier(wav2vec2_config_path).to(device)
model.eval()

print("✅ 微调的 BERT + Wav2Vec2 模型加载成功！")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of the model checkpoint at facebook/wav2vec2-base were not used when initializing Wav2Vec2ForPreTraining: ['wav2vec2.encoder.pos_conv_embed.conv.weight_v', 'wav2vec2.encoder.pos_conv_embed.conv.weight_g']
- This IS expected if you are initializing Wav2Vec2ForPreTraining from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForPreTraining from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weigh

✅ 微调的 BERT + Wav2Vec2 模型加载成功！


In [4]:
from safetensors.torch import load_file
try:
    # **加载 safetensors**
    # 本地加载
    safetensors_path = "E:\Github项目\web\model.safetensors"
    # huggingface_hub 仓库下载
    # model_path = hf_hub_download(repo_id="liloge/Group7_model_test", filename="model.safetensors")
    state_dict = load_file(safetensors_path)
    
    # **检查是否正确加载**
    print(f"成功加载 `{safetensors_path}`，包含 {len(state_dict.keys())} 个参数")
    
    # **打印部分参数**
    for key in list(state_dict.keys())[:5]:
        print(f" - {key}: {state_dict[key].shape}")
    try:
        # **确保权重匹配**
        model.load_state_dict(state_dict, strict=True)
        print("✅ safetensors 加载成功！")
    except Exception as e:
        missing_keys = set(model.state_dict().keys()) - set(state_dict.keys())
        print("Missing keys:", missing_keys)
        print(f"❌ safetensors 加载失败: {e}")
        # 检查缺失的参数
        exit(1)
except Exception as e:
    print(f"❌ 加载 `safetensors` 失败: {e}")
    exit(1)

成功加载 `E:\Github项目\web\model.safetensors`，包含 425 个参数
 - bert.bert.embeddings.LayerNorm.bias: torch.Size([768])
 - bert.bert.embeddings.LayerNorm.weight: torch.Size([768])
 - bert.bert.embeddings.position_embeddings.weight: torch.Size([512, 768])
 - bert.bert.embeddings.token_type_embeddings.weight: torch.Size([2, 768])
 - bert.bert.embeddings.word_embeddings.weight: torch.Size([30522, 768])
✅ safetensors 加载成功！


In [3]:
from safetensors.torch import load_file

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# **定义路径**
# bert_ckpt_path = "bert_meld_finetune_model.pth"
wav2vec2_config_path = "config.json"
# wav2vec2_safetensors_path = "wav2vec2.safetensors"

# **加载模型**
model = MultimodalClassifier(wav2vec2_config_path).to(device)
# 加载当前模型的 state_dict
current_state_dict = model.state_dict()

# 加载训练时保存的 safetensor 权重
safetensor_state_dict = load_file("E:\Github项目\web\model.safetensors")

# 检查当前模型中缺失的键
for key in current_state_dict.keys():
    if key not in safetensor_state_dict:
        print(f"❌ 缺失参数: {key}")

# 检查 safetensor 中有多余的键
for key in safetensor_state_dict.keys():
    if key not in current_state_dict:
        print(f"❌ 多余参数: {key}")
        
try:
    model.load_state_dict(safetensor_state_dict, strict=True)
    print("✅ 模型架构完全一致，权重加载成功！")
except Exception as e:
    print(f"❌ 模型架构不一致: {e}")



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of the model checkpoint at facebook/wav2vec2-base were not used when initializing Wav2Vec2ForPreTraining: ['wav2vec2.encoder.pos_conv_embed.conv.weight_v', 'wav2vec2.encoder.pos_conv_embed.conv.weight_g']
- This IS expected if you are initializing Wav2Vec2ForPreTraining from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForPreTraining from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weigh

✅ 模型架构完全一致，权重加载成功！
